# Legacy Models Evaluation: HOG, CNN, and ViT

This notebook evaluates the **legacy face recognition models** (HOG, CNN, ViT) using the same
metrics and methodology as the InsightFace baseline evaluation notebook.

**Models Evaluated:**
- **HOG** — `face_recognition` library with HOG face detector
- **CNN** — `face_recognition` library with CNN face detector
- **ViT** — Vision Transformer (ViT-B/32) with buffalo_l face detector

**Metrics Calculated (same as InsightFace evaluation):**
- Subset Accuracy, F-beta (macro/micro), Precision/Recall (macro/micro)
- Precision@Recall=0.95 (macro/micro), Recall@Precision=0.95 (macro/micro)
- Per-class: Precision, Recall, F-beta, ROC-AUC, P@R, R@P, confusion values
- PR curves, ROC curves, heatmaps

In [1]:
import datetime
import json
import copy
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Add project root to path and change working directory so relative image paths work
sys.path.insert(0, str(Path("/app")))
os.chdir("/app")

from src.classification import get_classifier, load_celebrities_from_json

BASE_DIR = Path("/app")
TRAINSET_PATH = BASE_DIR / "testsets/four-people-trainset-sample.json"
REFERENCES_PATH = BASE_DIR / "data/references.json"
OUTPUT_ROOT = BASE_DIR / "image_outputs"

# Create experiment output directory
EXPERIMENT_DIR = OUTPUT_ROOT / f"legacy_models_eval_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

# Evaluation parameters — identical to the InsightFace evaluation notebook
FACE_IDENTIFICATION_THRESHOLD = 0.30  # Min similarity to identify a face
F1_BETA = 0.4                         # F-beta parameter (< 1 emphasizes precision)
FIXED_RECALL_LEVEL = 0.95
FIXED_PRECISION_LEVEL = 0.95

# Legacy model configurations to evaluate
# Check if ViT/CLIP libraries are available
try:
    from transformers import CLIPProcessor, CLIPModel
    _VIT_AVAILABLE = True
except ImportError:
    _VIT_AVAILABLE = False

LEGACY_MODELS = {
    "face_recognition_hog": {
        "display_name": "HOG (face_recognition)",
        "classifier_type": "face_recognition_hog",
    },
    "face_recognition_cnn": {
        "display_name": "CNN (face_recognition)",
        "classifier_type": "face_recognition_cnn",
    },
}

if _VIT_AVAILABLE:
    LEGACY_MODELS["vit_b32"] = {
        "display_name": "ViT-B/32",
        "classifier_type": "vit_b32",
    }
else:
    print("⚠ ViT/CLIP libraries not installed — skipping ViT-B/32 evaluation")

print(f"✓ Setup complete")
print(f"\nPaths:")
print(f"  Output directory: {EXPERIMENT_DIR}")
print(f"  Training set: {TRAINSET_PATH}")
print(f"  References: {REFERENCES_PATH}")
print(f"  Working directory: {os.getcwd()}")
print(f"\nEvaluation Configuration:")
print(f"  Face identification threshold: {FACE_IDENTIFICATION_THRESHOLD}")
print(f"  F-beta parameter: {F1_BETA}")
print(f"  Fixed recall level: {FIXED_RECALL_LEVEL}")
print(f"  Fixed precision level: {FIXED_PRECISION_LEVEL}")
print(f"\nModels to evaluate:")
for key, cfg in LEGACY_MODELS.items():
    print(f"  - {cfg['display_name']} ({key})")

/opt/venv/lib/python3.12/site-packages/face_recognition_models/__init__.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
/opt/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


⚠ ViT/CLIP libraries not installed — skipping ViT-B/32 evaluation
✓ Setup complete

Paths:
  Output directory: /app/image_outputs/legacy_models_eval_20260214_153930
  Training set: /app/testsets/four-people-trainset-sample.json
  References: /app/data/references.json
  Working directory: /app

Evaluation Configuration:
  Face identification threshold: 0.3
  F-beta parameter: 0.4
  Fixed recall level: 0.95
  Fixed precision level: 0.95

Models to evaluate:
  - HOG (face_recognition) (face_recognition_hog)
  - CNN (face_recognition) (face_recognition_cnn)


## Load Training Set and References

In [2]:
# Load training set
with open(TRAINSET_PATH, "r", encoding="utf-8") as f:
    train_items = json.load(f)

print(f"✓ Loaded {len(train_items)} training images")

# Analyze training set distribution
label_counts = Counter()
for item in train_items:
    for label in item.get("labels", []):
        label_counts[label] += 1

print(f"\nTraining set label distribution:")
for label, count in sorted(label_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {label}: {count} images")

# Load celebrity reference data (normalised by load_celebrities_from_json)
celebrity_data = load_celebrities_from_json(str(REFERENCES_PATH))
print(f"\n✓ Loaded {len(celebrity_data)} celebrity reference(s)")
for cd in celebrity_data:
    print(f"  - {cd['name']}: {cd['reference_image_path']}")

# Also load raw references for identity list
with open(REFERENCES_PATH, "r", encoding="utf-8") as f:
    base_references = json.load(f)
all_identities = [ref["name"] for ref in base_references]
print(f"\nIdentities: {all_identities}")

2026-02-14 15:39:35,797 - src.logging_utils - INFO - Loading celebrity data from /app/data/references.json
2026-02-14 15:39:35,801 - src.logging_utils - INFO - Successfully loaded 4 celebrity reference(s) from /app/data/references.json


✓ Loaded 307 training images

Training set label distribution:
  Lionel Messi: 73 images
  Hugh Jackman: 70 images
  Donald Trump: 70 images
  Giorgia Meloni: 56 images
  None: 48 images

✓ Loaded 4 celebrity reference(s)
  - Hugh Jackman: Images/references/HughJackman.jpg
  - Donald Trump: Images/references/DonaldTrump.jpg
  - Giorgia Meloni: Images/references/GiorgiaMeloni.jpg
  - Lionel Messi: Images/references/LionnelMessi.png

Identities: ['Hugh Jackman', 'Donald Trump', 'Giorgia Meloni', 'Lionel Messi']


## Evaluate Legacy Models

For each legacy model we:
1. Initialise the classifier (detection + embedding + matching pipeline)
2. Run inference on every training image
3. Collect per-identity similarity scores for proper PR/ROC curve computation
4. Compute the full suite of metrics

In [3]:
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, fbeta_score,
    precision_recall_curve, roc_curve, auc as sk_auc,
)


def evaluate_model_with_scores(classifier, train_items, all_identities, threshold):
    """
    Run classifier on all images and return predictions + per-identity scores.
    
    Unlike the simple classify_images call, this method collects the raw
    similarity scores for EVERY identity (not just the best match) so that
    we can compute proper PR curves, ROC curves, and threshold-dependent
    metrics — exactly as is done in the InsightFace evaluation notebook.
    """
    all_predictions = []
    all_ground_truth = []
    all_prediction_scores = []

    total_faces = 0
    identified_faces = 0
    unknown_faces = 0
    no_face_count = 0

    ref_embeddings = np.array(classifier.reference_embeddings)
    ref_names = classifier.reference_names

    for item in tqdm(train_items, desc=f"Evaluating ({classifier.name})"):
        img_path_str = item.get("path", "")
        ground_truth = set(item.get("labels", []))

        predicted_set = set()
        image_scores = {identity: [] for identity in all_identities}

        img = cv2.imread(img_path_str)
        if img is None:
            no_face_count += 1
            predicted_set.add("None")
            all_ground_truth.append(list(ground_truth))
            all_predictions.append(list(predicted_set))
            all_prediction_scores.append({identity: 0.0 for identity in all_identities})
            continue

        # Phase 1: Detect faces
        face_bboxes = classifier.face_detector.detect_faces(img)
        if not face_bboxes:
            no_face_count += 1
            predicted_set.add("None")
            all_ground_truth.append(list(ground_truth))
            all_predictions.append(list(predicted_set))
            all_prediction_scores.append({identity: 0.0 for identity in all_identities})
            continue

        # Phase 2: Extract embeddings
        if (classifier.use_alignment
            and hasattr(classifier, 'embedder')
            and hasattr(classifier.embedder, 'extract_embedding_aligned')
            and classifier.face_detector.model in classifier.face_detector.INSIGHTFACE_MODELS):
            face_infos = classifier.face_detector.detect_faces_with_landmarks(img)
            embeddings = [classifier.embedder.extract_embedding_aligned(img, fi) for fi in face_infos]
        else:
            embeddings = classifier.embedder.extract_embeddings_batch(img, face_bboxes)

        for embedding in embeddings:
            if embedding is None:
                continue
            total_faces += 1

            # Phase 3: Compute similarity against every reference
            identity_scores = {}
            for ref_emb, ref_name in zip(ref_embeddings, ref_names):
                score = float(classifier.matcher.compare(embedding, ref_emb))
                # Keep best score per identity (supports multiple ref images)
                if ref_name not in identity_scores or score > identity_scores[ref_name]:
                    identity_scores[ref_name] = score

            for identity in all_identities:
                s = identity_scores.get(identity, 0.0)
                image_scores[identity].append(s)

            # Identify using threshold
            best_match = max(identity_scores, key=identity_scores.get) if identity_scores else None
            best_score = identity_scores.get(best_match, 0.0) if best_match else 0.0
            if best_score >= threshold:
                predicted_set.add(best_match)
                identified_faces += 1
            else:
                unknown_faces += 1

        if not predicted_set:
            predicted_set.add("None")

        all_ground_truth.append(list(ground_truth))
        all_predictions.append(list(predicted_set))

        # Max score per identity across all faces in image
        max_scores = {identity: max(scores) if scores else 0.0
                      for identity, scores in image_scores.items()}
        all_prediction_scores.append(max_scores)

    return {
        "all_predictions": all_predictions,
        "all_ground_truth": all_ground_truth,
        "all_prediction_scores": all_prediction_scores,
        "total_faces": total_faces,
        "identified_faces": identified_faces,
        "unknown_faces": unknown_faces,
        "no_face_count": no_face_count,
    }


def compute_full_metrics(eval_result, all_identities, f_beta, fixed_recall, fixed_precision):
    """Compute the same comprehensive metrics as the InsightFace evaluation notebook."""
    all_ground_truth = eval_result["all_ground_truth"]
    all_predictions = eval_result["all_predictions"]
    all_prediction_scores = eval_result["all_prediction_scores"]

    # Multi-label binarisation
    mlb = MultiLabelBinarizer()
    y_true = mlb.fit_transform(all_ground_truth)
    y_pred = mlb.transform(all_predictions)

    # Overall metrics
    subset_accuracy = float(accuracy_score(y_true, y_pred))
    f_beta_macro = float(fbeta_score(y_true, y_pred, beta=f_beta, average='macro', zero_division=0))
    f_beta_micro = float(fbeta_score(y_true, y_pred, beta=f_beta, average='micro', zero_division=0))
    precision_macro = float(precision_score(y_true, y_pred, average='macro', zero_division=0))
    precision_micro = float(precision_score(y_true, y_pred, average='micro', zero_division=0))
    recall_macro = float(recall_score(y_true, y_pred, average='macro', zero_division=0))
    recall_micro = float(recall_score(y_true, y_pred, average='micro', zero_division=0))

    # ── Per-class metrics ──────────────────────────────────────────────────
    all_possible_labels = (set(c for sub in all_ground_truth for c in sub)
                           | set(c for sub in all_predictions for c in sub))
    mlb_full = MultiLabelBinarizer()
    mlb_full.fit([list(all_possible_labels)])
    y_true_full = mlb_full.transform(all_ground_truth)
    y_pred_full = mlb_full.transform(all_predictions)

    beta2 = f_beta ** 2
    per_class_metrics = {}
    for i, class_name in enumerate(mlb_full.classes_):
        tp = int(np.sum((y_pred_full[:, i] == 1) & (y_true_full[:, i] == 1)))
        fp = int(np.sum((y_pred_full[:, i] == 1) & (y_true_full[:, i] == 0)))
        fn = int(np.sum((y_pred_full[:, i] == 0) & (y_true_full[:, i] == 1)))
        tn = int(np.sum((y_pred_full[:, i] == 0) & (y_true_full[:, i] == 0)))

        prec = tp / (tp + fp) if (tp + fp) > 0 else 0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0
        acc = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0
        fb = ((1 + beta2) * prec * rec / (beta2 * prec + rec)) if (prec + rec) > 0 else 0

        # Scores for this class
        y_true_binary = y_true_full[:, i]
        if class_name == 'None':
            y_scores = np.array([1 - max(s.values()) if s else 1.0 for s in all_prediction_scores])
        else:
            y_scores = np.array([s.get(class_name, 0.0) for s in all_prediction_scores])

        prec_curve, rec_curve, _ = precision_recall_curve(y_true_binary, y_scores)
        p_at_fixed_r = float(np.interp(fixed_recall, rec_curve[::-1], prec_curve[::-1], left=0.0, right=0.0))

        valid_recalls = rec_curve[prec_curve >= fixed_precision]
        r_at_fixed_p = float(valid_recalls.max()) if len(valid_recalls) > 0 else 0.0

        fpr, tpr, _ = roc_curve(y_true_binary, y_scores)
        roc_auc = sk_auc(fpr, tpr)

        per_class_metrics[class_name] = {
            "precision": float(prec), "recall": float(rec), "f_beta": float(fb),
            "accuracy": float(acc), "support": tp + fn,
            "tp": tp, "fp": fp, "fn": fn, "tn": tn,
            "precision_at_fixed_recall": p_at_fixed_r,
            "recall_at_fixed_precision": r_at_fixed_p,
            "roc_auc": float(roc_auc),
        }

    # ── Macro / micro P@R and R@P ──────────────────────────────────────────
    p_at_r_macro = np.mean([per_class_metrics[c]["precision_at_fixed_recall"]
                            for c in per_class_metrics])
    r_at_p_macro = np.mean([per_class_metrics[c]["recall_at_fixed_precision"]
                            for c in per_class_metrics])

    # micro-averaged PR curve
    y_true_flat = y_true.ravel()
    y_scores_flat = []
    for sample_idx in range(len(all_prediction_scores)):
        for class_idx, class_name in enumerate(mlb.classes_):
            if class_name == 'None':
                score = 1 - max(all_prediction_scores[sample_idx].values()) if all_prediction_scores[sample_idx] else 1.0
            else:
                score = all_prediction_scores[sample_idx].get(class_name, 0.0)
            y_scores_flat.append(score)
    y_scores_flat = np.array(y_scores_flat)

    prec_micro_curve, rec_micro_curve, _ = precision_recall_curve(y_true_flat, y_scores_flat)
    p_at_r_micro = float(np.interp(fixed_recall, rec_micro_curve[::-1], prec_micro_curve[::-1], left=0.0, right=0.0))
    valid_micro = rec_micro_curve[prec_micro_curve >= fixed_precision]
    r_at_p_micro = float(valid_micro.max()) if len(valid_micro) > 0 else 0.0

    total_faces = eval_result["total_faces"]
    identified = eval_result["identified_faces"]

    result = {
        "total_faces": total_faces,
        "identified_faces": identified,
        "unknown_faces": eval_result["unknown_faces"],
        "identification_rate": float(identified / max(total_faces, 1)),
        "subset_accuracy": subset_accuracy,
        "f_beta_macro": f_beta_macro,
        "f_beta_micro": f_beta_micro,
        "precision_macro": precision_macro,
        "precision_micro": precision_micro,
        "recall_macro": recall_macro,
        "recall_micro": recall_micro,
        "precision_at_fixed_recall_macro": float(p_at_r_macro),
        "precision_at_fixed_recall_micro": float(p_at_r_micro),
        "recall_at_fixed_precision_macro": float(r_at_p_macro),
        "recall_at_fixed_precision_micro": float(r_at_p_micro),
    }

    detailed = {
        "y_true": y_true,
        "y_pred": y_pred,
        "scores": all_prediction_scores,
        "ground_truth": all_ground_truth,
        "predictions": all_predictions,
        "class_names": mlb.classes_.tolist(),
        "per_class_metrics": per_class_metrics,
    }

    return result, detailed

print("✓ Evaluation functions defined")

✓ Evaluation functions defined


## Run Evaluation — HOG, CNN, ViT

Each model is initialised, run on the full training set, and evaluated.
This cell may take several minutes per model (especially CNN and ViT).

In [4]:
# Storage for all model results
model_results = {}      # model_name -> summary dict
model_detailed = {}     # model_name -> detailed data (y_true, y_pred, scores, etc.)

for model_key, model_cfg in LEGACY_MODELS.items():
    display = model_cfg["display_name"]
    ctype = model_cfg["classifier_type"]

    print(f"\n{'=' * 80}")
    print(f"EVALUATING: {display} ({ctype})")
    print(f"{'=' * 80}")

    # Initialise classifier
    try:
        classifier = get_classifier(ctype, celebrity_data)
    except Exception as e:
        print(f"⚠ Could not initialise {display}: {e}")
        print(f"  Skipping this model.")
        continue

    print(f"✓ Classifier initialised: {classifier.name}")
    print(f"  Reference embeddings: {len(classifier.reference_embeddings)}")

    if len(classifier.reference_embeddings) == 0:
        print(f"⚠ No reference embeddings — skipping evaluation (check reference image paths)")
        del classifier
        continue

    # Run evaluation with per-identity scores
    eval_result = evaluate_model_with_scores(
        classifier, train_items, all_identities,
        threshold=FACE_IDENTIFICATION_THRESHOLD,
    )

    print(f"\n  Total faces detected: {eval_result['total_faces']}")
    print(f"  Identified: {eval_result['identified_faces']}")
    print(f"  Unknown: {eval_result['unknown_faces']}")
    print(f"  No face: {eval_result['no_face_count']}")

    # Compute full metrics
    result, detailed = compute_full_metrics(
        eval_result, all_identities,
        f_beta=F1_BETA,
        fixed_recall=FIXED_RECALL_LEVEL,
        fixed_precision=FIXED_PRECISION_LEVEL,
    )
    result["model"] = model_key
    result["display_name"] = display

    model_results[model_key] = result
    model_detailed[model_key] = detailed

    # Print summary
    print(f"\n  ── Results ──")
    print(f"  Subset Accuracy:         {result['subset_accuracy']:.4f}")
    print(f"  F-beta (macro, β={F1_BETA}):  {result['f_beta_macro']:.4f}")
    print(f"  F-beta (micro, β={F1_BETA}):  {result['f_beta_micro']:.4f}")
    print(f"  Precision (macro):       {result['precision_macro']:.4f}")
    print(f"  Precision (micro):       {result['precision_micro']:.4f}")
    print(f"  Recall (macro):          {result['recall_macro']:.4f}")
    print(f"  Recall (micro):          {result['recall_micro']:.4f}")
    print(f"  P@R={FIXED_RECALL_LEVEL} (macro):      {result['precision_at_fixed_recall_macro']:.4f}")
    print(f"  P@R={FIXED_RECALL_LEVEL} (micro):      {result['precision_at_fixed_recall_micro']:.4f}")
    print(f"  R@P={FIXED_PRECISION_LEVEL} (macro):      {result['recall_at_fixed_precision_macro']:.4f}")
    print(f"  R@P={FIXED_PRECISION_LEVEL} (micro):      {result['recall_at_fixed_precision_micro']:.4f}")

    # Free memory
    del classifier

print(f"\n{'=' * 80}")
print(f"EVALUATIONS COMPLETE — {len(model_results)} model(s) evaluated")
print(f"{'=' * 80}")

2026-02-14 15:40:05,123 - src.logging_utils - INFO - Getting classifier: face_recognition_hog
2026-02-14 15:40:05,124 - src.logging_utils - INFO - Initializing UnifiedClassifier: face_recognition_hog
  Detection: hog
  Embedding: face_recognition
  Matching: euclidean_distance
  Threshold: 0.6
2026-02-14 15:40:05,125 - src.logging_utils - INFO - FaceDetector initialized with model: hog, det_size: (640, 640), multi-pass: False



EVALUATING: HOG (face_recognition) (face_recognition_hog)


2026-02-14 15:40:08,933 - src.logging_utils - INFO - Successfully initialized face_recognition_hog with 4 reference embeddings


✓ Classifier initialised: face_recognition_hog
  Reference embeddings: 4


Evaluating (face_recognition_hog): 100%|██████████| 307/307 [02:07<00:00,  2.41it/s]
2026-02-14 15:42:16,565 - src.logging_utils - INFO - Getting classifier: face_recognition_cnn
2026-02-14 15:42:16,567 - src.logging_utils - INFO - Initializing UnifiedClassifier: face_recognition_cnn
  Detection: cnn
  Embedding: face_recognition
  Matching: euclidean_distance
  Threshold: 0.6
2026-02-14 15:42:16,567 - src.logging_utils - INFO - FaceDetector initialized with model: cnn, det_size: (640, 640), multi-pass: True



  Total faces detected: 464
  Identified: 464
  Unknown: 0
  No face: 83

  ── Results ──
  Subset Accuracy:         0.4984
  F-beta (macro, β=0.4):  0.5500
  F-beta (micro, β=0.4):  0.5437
  Precision (macro):       0.5361
  Precision (micro):       0.5270
  Recall (macro):          0.6731
  Recall (micro):          0.6782
  P@R=0.95 (macro):      0.2473
  P@R=0.95 (micro):      0.2092
  R@P=0.95 (macro):      0.4845
  R@P=0.95 (micro):      0.0000

EVALUATING: CNN (face_recognition) (face_recognition_cnn)


: 

## Total Evaluation Metrics — Comparison Table

In [ ]:
# Build comparison DataFrame
rows = []
for model_key in model_results:
    r = model_results[model_key]
    rows.append(r)

results_df = pd.DataFrame(rows)

print("=" * 120)
print("LEGACY MODEL EVALUATION - COMPARISON TABLE")
print("=" * 120)
print(f"\nConfiguration:")
print(f"  F-beta parameter (β): {F1_BETA}")
print(f"  Face identification threshold: {FACE_IDENTIFICATION_THRESHOLD}")
print(f"  Fixed recall level: {FIXED_RECALL_LEVEL}")
print(f"  Fixed precision level: {FIXED_PRECISION_LEVEL}")
print(f"\n{'─' * 120}\n")

display_df = results_df[[
    'display_name', 'subset_accuracy',
    'f_beta_macro', 'f_beta_micro',
    'precision_macro', 'precision_micro',
    'recall_macro', 'recall_micro',
    'precision_at_fixed_recall_macro', 'precision_at_fixed_recall_micro',
    'recall_at_fixed_precision_macro', 'recall_at_fixed_precision_micro',
]].copy()

display_df.columns = [
    'Model', 'Subset Acc',
    f'F-β (macro)', f'F-β (micro)',
    'Prec (macro)', 'Prec (micro)',
    'Rec (macro)', 'Rec (micro)',
    f'P@R={FIXED_RECALL_LEVEL} (ma)', f'P@R={FIXED_RECALL_LEVEL} (mi)',
    f'R@P={FIXED_PRECISION_LEVEL} (ma)', f'R@P={FIXED_PRECISION_LEVEL} (mi)',
]

# Format numeric columns
for col in display_df.columns[1:]:
    display_df[col] = display_df[col].map(lambda x: f"{x:.4f}")

print(display_df.to_string(index=False))
print(f"\n{'=' * 120}")

## Per-Class Performance — All Models

In [ ]:
print("=" * 120)
print("PER-CLASS METRICS BY MODEL")
print("=" * 120)

for model_key in model_results:
    display = LEGACY_MODELS[model_key]["display_name"]
    pcm = model_detailed[model_key]["per_class_metrics"]

    print(f"\n{'─' * 100}")
    print(f"  {display}")
    print(f"{'─' * 100}")
    print(f"  {'Class':<25} {'Prec':>8} {'Recall':>8} {'F-beta':>8} {'ROC-AUC':>8} {'P@R=.95':>8} {'R@P=.95':>8} {'TP':>6} {'FP':>6} {'FN':>6} {'TN':>6}")
    print(f"  {'─' * 96}")
    for cn in sorted(pcm.keys()):
        m = pcm[cn]
        print(f"  {cn:<25} {m['precision']:>8.4f} {m['recall']:>8.4f} {m['f_beta']:>8.4f} "
              f"{m['roc_auc']:>8.4f} {m['precision_at_fixed_recall']:>8.4f} "
              f"{m['recall_at_fixed_precision']:>8.4f} {m['tp']:>6} {m['fp']:>6} {m['fn']:>6} {m['tn']:>6}")

print(f"\n{'=' * 120}")

## Per-Class Heatmaps

In [ ]:
# Per-class metric heatmaps for each model
metrics_for_heatmap = ['precision', 'recall', 'f_beta', 'roc_auc']
metric_labels = ['Precision', 'Recall', f'F-beta (β={F1_BETA})', 'ROC-AUC']

evaluated_models = list(model_results.keys())
n_models = len(evaluated_models)
fig, axes = plt.subplots(n_models, len(metrics_for_heatmap), figsize=(20, 4 * n_models))
if n_models == 1:
    axes = axes[np.newaxis, :]

for row_idx, model_key in enumerate(evaluated_models):
    pcm = model_detailed[model_key]["per_class_metrics"]
    display = LEGACY_MODELS[model_key]["display_name"]
    class_names_sorted = sorted(pcm.keys())

    for col_idx, (metric_key, metric_label) in enumerate(zip(metrics_for_heatmap, metric_labels)):
        ax = axes[row_idx, col_idx]
        values = [[pcm[cn][metric_key] for cn in class_names_sorted]]
        sns.heatmap(
            values, annot=True, fmt=".3f", cmap="YlOrRd",
            xticklabels=class_names_sorted, yticklabels=[display],
            ax=ax, vmin=0, vmax=1, cbar=col_idx == len(metrics_for_heatmap) - 1,
        )
        if row_idx == 0:
            ax.set_title(metric_label, fontsize=12, fontweight='bold')

fig.suptitle("Per-Class Metrics Heatmap — Legacy Models", fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
heatmap_path = EXPERIMENT_DIR / "per_class_metrics_heatmaps.png"
plt.savefig(heatmap_path, dpi=150, bbox_inches='tight')
print(f"✓ Saved: {heatmap_path}")
plt.show()

## Macro-Averaged PR and ROC Curves — Per Model

In [ ]:
evaluated_models = list(model_results.keys())
n_models = len(evaluated_models)
fig_pr, axes_pr = plt.subplots(1, n_models, figsize=(8 * n_models, 7))
fig_roc, axes_roc = plt.subplots(1, n_models, figsize=(8 * n_models, 7))

if n_models == 1:
    axes_pr = [axes_pr]
    axes_roc = [axes_roc]

colors_palette = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#8e44ad']

for idx, model_key in enumerate(evaluated_models):
    display = LEGACY_MODELS[model_key]["display_name"]
    det = model_detailed[model_key]
    y_true = det["y_true"]
    scores = det["scores"]
    class_names = det["class_names"]

    all_precisions = []
    all_recalls = []
    all_fprs = []
    all_tprs = []
    class_aucs_pr = []
    class_aucs_roc = []

    for class_idx, class_name in enumerate(class_names):
        y_true_binary = y_true[:, class_idx]
        if class_name == 'None':
            y_scores = np.array([1 - max(s.values()) if s else 1.0 for s in scores])
        else:
            y_scores = np.array([s.get(class_name, 0.0) for s in scores])

        prec, rec, _ = precision_recall_curve(y_true_binary, y_scores)
        pr_auc = sk_auc(rec, prec)
        all_precisions.append(prec)
        all_recalls.append(rec)
        class_aucs_pr.append(pr_auc)

        fpr, tpr, _ = roc_curve(y_true_binary, y_scores)
        roc_auc_val = sk_auc(fpr, tpr)
        all_fprs.append(fpr)
        all_tprs.append(tpr)
        class_aucs_roc.append(roc_auc_val)

    # ── PR curve ──
    ax_pr = axes_pr[idx]
    mean_recall = np.linspace(0, 1, 200)
    mean_precision = np.zeros_like(mean_recall)
    for p, r in zip(all_precisions, all_recalls):
        mean_precision += np.interp(mean_recall, r[::-1], p[::-1])
    mean_precision /= len(all_precisions)
    macro_auc_pr = sk_auc(mean_recall, mean_precision)

    ax_pr.plot(mean_recall, mean_precision, linewidth=3, color='navy',
               label=f'Macro-avg (AUC={macro_auc_pr:.3f})')
    ax_pr.fill_between(mean_recall, mean_precision, alpha=0.15, color='navy')
    for ci, (p, r, cn, au) in enumerate(zip(all_precisions, all_recalls, class_names, class_aucs_pr)):
        ax_pr.plot(r, p, linewidth=1.5, alpha=0.5,
                   color=colors_palette[ci % len(colors_palette)],
                   label=f'{cn} (AUC={au:.3f})')
    ax_pr.set_xlabel('Recall', fontsize=11, fontweight='bold')
    ax_pr.set_ylabel('Precision', fontsize=11, fontweight='bold')
    ax_pr.set_title(f'{display}\nMacro PR-AUC: {macro_auc_pr:.3f}', fontsize=13, fontweight='bold')
    ax_pr.set_xlim([0, 1]); ax_pr.set_ylim([0, 1.05])
    ax_pr.legend(fontsize=8, loc='best')
    ax_pr.grid(True, alpha=0.3, linestyle='--')

    # ── ROC curve ──
    ax_roc = axes_roc[idx]
    mean_fpr = np.linspace(0, 1, 200)
    mean_tpr = np.zeros_like(mean_fpr)
    for f, t in zip(all_fprs, all_tprs):
        mean_tpr += np.interp(mean_fpr, f, t)
    mean_tpr /= len(all_fprs)
    mean_tpr[0] = 0.0
    macro_auc_roc = sk_auc(mean_fpr, mean_tpr)

    ax_roc.plot(mean_fpr, mean_tpr, linewidth=3, color='navy',
                label=f'Macro-avg (AUC={macro_auc_roc:.3f})')
    ax_roc.fill_between(mean_fpr, mean_tpr, alpha=0.15, color='navy')
    ax_roc.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Random')
    for ci, (f, t, cn, au) in enumerate(zip(all_fprs, all_tprs, class_names, class_aucs_roc)):
        ax_roc.plot(f, t, linewidth=1.5, alpha=0.5,
                    color=colors_palette[ci % len(colors_palette)],
                    label=f'{cn} (AUC={au:.3f})')
    ax_roc.set_xlabel('FPR', fontsize=11, fontweight='bold')
    ax_roc.set_ylabel('TPR', fontsize=11, fontweight='bold')
    ax_roc.set_title(f'{display}\nMacro ROC-AUC: {macro_auc_roc:.3f}', fontsize=13, fontweight='bold')
    ax_roc.set_xlim([0, 1]); ax_roc.set_ylim([0, 1.05])
    ax_roc.legend(fontsize=8, loc='lower right')
    ax_roc.grid(True, alpha=0.3, linestyle='--')

fig_pr.suptitle("Precision-Recall Curves — Legacy Models", fontsize=16, fontweight='bold', y=1.02)
fig_roc.suptitle("ROC Curves — Legacy Models", fontsize=16, fontweight='bold', y=1.02)

plt.figure(fig_pr.number)
pr_path = EXPERIMENT_DIR / "pr_curves_all_models.png"
plt.savefig(pr_path, dpi=150, bbox_inches='tight')
print(f"✓ PR curves saved: {pr_path}")

plt.figure(fig_roc.number)
roc_path = EXPERIMENT_DIR / "roc_curves_all_models.png"
plt.savefig(roc_path, dpi=150, bbox_inches='tight')
print(f"✓ ROC curves saved: {roc_path}")

plt.show()

## Combined ROC Curve — All Models on One Plot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))

model_colors = {'face_recognition_hog': '#e74c3c', 'face_recognition_cnn': '#3498db', 'vit_b32': '#2ecc71'}

for model_key in model_results:
    display = LEGACY_MODELS[model_key]["display_name"]
    det = model_detailed[model_key]
    y_true = det["y_true"]
    scores = det["scores"]
    class_names = det["class_names"]

    mean_fpr = np.linspace(0, 1, 200)
    mean_tpr = np.zeros_like(mean_fpr)

    for class_idx, class_name in enumerate(class_names):
        y_true_binary = y_true[:, class_idx]
        if class_name == 'None':
            y_scores = np.array([1 - max(s.values()) if s else 1.0 for s in scores])
        else:
            y_scores = np.array([s.get(class_name, 0.0) for s in scores])
        fpr, tpr, _ = roc_curve(y_true_binary, y_scores)
        mean_tpr += np.interp(mean_fpr, fpr, tpr)

    mean_tpr /= len(class_names)
    mean_tpr[0] = 0.0
    auc_val = sk_auc(mean_fpr, mean_tpr)

    ax.plot(mean_fpr, mean_tpr, linewidth=3, color=model_colors.get(model_key, 'gray'),
            label=f'{display} (Macro AUC={auc_val:.3f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1.2, alpha=0.5, label='Random Classifier')
ax.set_xlim([-0.02, 1.02]); ax.set_ylim([-0.02, 1.05])
ax.set_xlabel('False Positive Rate', fontsize=13, fontweight='bold')
ax.set_ylabel('True Positive Rate', fontsize=13, fontweight='bold')
ax.set_title('ROC Curve Comparison — Legacy Models', fontsize=14, fontweight='bold', pad=15)
ax.legend(loc='lower right', fontsize=11, framealpha=0.9, edgecolor='gray')
ax.grid(True, alpha=0.25, linestyle='--')

plt.tight_layout()
combined_roc_path = EXPERIMENT_DIR / "roc_curve_combined_legacy.png"
plt.savefig(combined_roc_path, dpi=150, bbox_inches='tight')
print(f"✓ Combined ROC curve saved: {combined_roc_path}")
plt.show()

## Combined PR Curve — All Models on One Plot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))

model_colors = {'face_recognition_hog': '#e74c3c', 'face_recognition_cnn': '#3498db', 'vit_b32': '#2ecc71'}

for model_key in model_results:
    display = LEGACY_MODELS[model_key]["display_name"]
    det = model_detailed[model_key]
    y_true = det["y_true"]
    scores = det["scores"]
    class_names = det["class_names"]

    mean_recall = np.linspace(0, 1, 200)
    mean_precision = np.zeros_like(mean_recall)

    for class_idx, class_name in enumerate(class_names):
        y_true_binary = y_true[:, class_idx]
        if class_name == 'None':
            y_scores = np.array([1 - max(s.values()) if s else 1.0 for s in scores])
        else:
            y_scores = np.array([s.get(class_name, 0.0) for s in scores])
        prec, rec, _ = precision_recall_curve(y_true_binary, y_scores)
        mean_precision += np.interp(mean_recall, rec[::-1], prec[::-1])

    mean_precision /= len(class_names)
    auc_val = sk_auc(mean_recall, mean_precision)

    ax.plot(mean_recall, mean_precision, linewidth=3, color=model_colors.get(model_key, 'gray'),
            label=f'{display} (Macro AUC={auc_val:.3f})')

ax.set_xlim([0, 1]); ax.set_ylim([0, 1.05])
ax.set_xlabel('Recall', fontsize=13, fontweight='bold')
ax.set_ylabel('Precision', fontsize=13, fontweight='bold')
ax.set_title('Precision-Recall Curve Comparison — Legacy Models', fontsize=14, fontweight='bold', pad=15)
ax.legend(loc='best', fontsize=11, framealpha=0.9, edgecolor='gray')
ax.grid(True, alpha=0.25, linestyle='--')

plt.tight_layout()
combined_pr_path = EXPERIMENT_DIR / "pr_curve_combined_legacy.png"
plt.savefig(combined_pr_path, dpi=150, bbox_inches='tight')
print(f"✓ Combined PR curve saved: {combined_pr_path}")
plt.show()

## Bar Chart Comparison — Key Metrics

In [ ]:
# Grouped bar chart comparing key metrics across models
compare_metrics = [
    ('subset_accuracy', 'Subset Accuracy'),
    ('f_beta_macro', f'F-β (macro, β={F1_BETA})'),
    ('precision_macro', 'Precision (macro)'),
    ('recall_macro', 'Recall (macro)'),
    ('precision_at_fixed_recall_macro', f'P@R={FIXED_RECALL_LEVEL}'),
    ('recall_at_fixed_precision_macro', f'R@P={FIXED_PRECISION_LEVEL}'),
]

model_colors = {'face_recognition_hog': '#e74c3c', 'face_recognition_cnn': '#3498db', 'vit_b32': '#2ecc71'}
model_keys = list(model_results.keys())
model_names = [LEGACY_MODELS[k]["display_name"] for k in model_keys]

x = np.arange(len(compare_metrics))
width = 0.8 / max(len(model_keys), 1)

fig, ax = plt.subplots(figsize=(16, 7))

for i, mk in enumerate(model_keys):
    values = [model_results[mk][cm[0]] for cm in compare_metrics]
    bars = ax.bar(x + i * width, values, width, label=model_names[i],
                  color=model_colors.get(mk, 'gray'), alpha=0.85)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_xticks(x + width * (len(model_keys) - 1) / 2)
ax.set_xticklabels([cm[1] for cm in compare_metrics], fontsize=10, rotation=15, ha='right')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Legacy Model Comparison — Key Metrics', fontsize=14, fontweight='bold')
ax.set_ylim([0, 1.15])
ax.legend(fontsize=11)
ax.grid(True, alpha=0.2, axis='y')

plt.tight_layout()
bar_path = EXPERIMENT_DIR / "model_comparison_bar_chart.png"
plt.savefig(bar_path, dpi=150, bbox_inches='tight')
print(f"✓ Bar chart saved: {bar_path}")
plt.show()

## Executive Summary Report

In [ ]:
evaluated_model_keys = list(model_results.keys())
n_evaluated = len(evaluated_model_keys)

print("\n" + "=" * 110)
print("LEGACY MODELS EVALUATION — EXECUTIVE SUMMARY REPORT")
print("=" * 110)

print(f"""
Configuration:
  Models evaluated:                {n_evaluated}
  Test Set:                        {len(train_items)} training images
  Face Identification Threshold:   {FACE_IDENTIFICATION_THRESHOLD}
  F-beta Parameter:                {F1_BETA}
  Fixed Recall Level:              {FIXED_RECALL_LEVEL}
  Fixed Precision Level:           {FIXED_PRECISION_LEVEL}
""")

# Overall metrics table
header = f"  {'Metric':<40}"
for mk in evaluated_model_keys:
    header += f"{LEGACY_MODELS[mk]['display_name']:>25}"
print(header)
print(f"  {'─' * (40 + 25 * n_evaluated)}")

metric_rows = [
    ('subset_accuracy', 'Subset Accuracy'),
    ('f_beta_macro', f'F-beta (macro, β={F1_BETA})'),
    ('f_beta_micro', f'F-beta (micro, β={F1_BETA})'),
    ('precision_macro', 'Precision (macro)'),
    ('precision_micro', 'Precision (micro)'),
    ('recall_macro', 'Recall (macro)'),
    ('recall_micro', 'Recall (micro)'),
    ('precision_at_fixed_recall_macro', f'P@R={FIXED_RECALL_LEVEL} (macro)'),
    ('precision_at_fixed_recall_micro', f'P@R={FIXED_RECALL_LEVEL} (micro)'),
    ('recall_at_fixed_precision_macro', f'R@P={FIXED_PRECISION_LEVEL} (macro)'),
    ('recall_at_fixed_precision_micro', f'R@P={FIXED_PRECISION_LEVEL} (micro)'),
    ('identification_rate', 'Identification Rate'),
]

for mkey, mlabel in metric_rows:
    row = f"  {mlabel:<40}"
    for mk in evaluated_model_keys:
        val = model_results[mk][mkey]
        row += f"{val:>25.4f}"
    print(row)

# Per-class summary
print(f"\n{'─' * (40 + 25 * n_evaluated)}")
print(f"  PER-CLASS F-BETA SCORES")
print(f"{'─' * (40 + 25 * n_evaluated)}")
header = f"  {'Class':<40}"
for mk in evaluated_model_keys:
    header += f"{LEGACY_MODELS[mk]['display_name']:>25}"
print(header)
print(f"  {'─' * (40 + 25 * n_evaluated)}")

all_classes = set()
for mk in evaluated_model_keys:
    all_classes.update(model_detailed[mk]["per_class_metrics"].keys())

for cn in sorted(all_classes):
    row = f"  {cn:<40}"
    for mk in evaluated_model_keys:
        pcm = model_detailed[mk]["per_class_metrics"]
        val = pcm.get(cn, {}).get('f_beta', 0.0)
        row += f"{val:>25.4f}"
    print(row)

# Best model per metric
print(f"\n{'─' * (40 + 25 * n_evaluated)}")
print(f"  BEST MODEL PER METRIC")
print(f"{'─' * (40 + 25 * n_evaluated)}")
for mkey, mlabel in metric_rows:
    best_model = max(evaluated_model_keys, key=lambda mk: model_results[mk][mkey])
    best_val = model_results[best_model][mkey]
    best_display = LEGACY_MODELS[best_model]["display_name"]
    print(f"  {mlabel:<40} {best_display:<25} {best_val:.4f}")

print(f"""
GENERATED VISUALIZATIONS:
  1. per_class_metrics_heatmaps.png     - Per-class metrics heatmap for all models
  2. pr_curves_all_models.png           - Per-model PR curves with per-class breakdown
  3. roc_curves_all_models.png          - Per-model ROC curves with per-class breakdown
  4. roc_curve_combined_legacy.png      - Combined macro-averaged ROC curves
  5. pr_curve_combined_legacy.png       - Combined macro-averaged PR curves
  6. model_comparison_bar_chart.png     - Key metrics bar chart comparison

All results saved to: {EXPERIMENT_DIR}
""")

print("=" * 110)
print("LEGACY MODELS EVALUATION COMPLETE")
print("=" * 110)